In [1]:
%pip install python-dotenv openai azure-ai-projects azure-identity azure-ai-inference azure-ai-ml 
%pip install openpyxl

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [4]:
import os
from dotenv import load_dotenv

#Setting connection details for CIS LLM (NON PROD US)
lite_llm_endpoint = "https://llm-api-cis.azure-intlsd-np.nielsencsp.net/"

load_dotenv()
api_key = os.getenv("CIS_API_KEY")
print("Loaded:", api_key[:5] + "*****")

Loaded: sk-z5*****


In [5]:
#Importing required packages
from azure.ai.inference import ChatCompletionsClient
from azure.core.credentials import AzureKeyCredential

#Setting up the chat client
client = ChatCompletionsClient(
    endpoint=f"{lite_llm_endpoint}",
    credential=AzureKeyCredential(api_key),
    api_version="2025-03-01-preview"
)

In [6]:
from azure.ai.inference.models import SystemMessage, UserMessage

models = ["hack-fest-gpt-5.6-luna"]          # Must match models deployed in AI Foundry

for model in models:
    try:
        response = client.complete(
            messages=[
                # SystemMessage(content="You are a helpful assistant with expert knowledge in marketing and consumer sales."),
                UserMessage(content="Generate a simple python code"),
            ],
            model=model,
            headers={"Authorization": api_key, }
        )

        print(f"Response from model {model}:\n\r{response.choices[0].message.content}\n\r")
        print(f"#####################################################################\n\r")
    except Exception as e:
        print(f"Error calling model {model}")
        print(f"Error type: {type(e).__name__}")
        print(f"Error message: {str(e)}")
        print(f"Endpoint: {lite_llm_endpoint}")
        print(f"API Key set: {'Yes' if api_key != 'nokey' else 'No - using placeholder'}")
        print(f"Model: {model}")


Response from model hack-fest-gpt-5.6-luna:
```python
print("Hello, world!")
```

#####################################################################



In [7]:
from azure.ai.inference.models import UserMessage

def ask_llm(prompt): 
    response = client.complete( messages=[ UserMessage(content=prompt) ], 
                                model="hack-fest-gpt-5.6-luna" ) 
    return response.choices[0].message.content

print(ask_llm("What is Biscuit?"))

A **biscuit** is a small baked food, typically made from flour and butter or another fat. In the U.S., it’s usually soft and flaky, often served with breakfast or alongside savory dishes. In the U.K. and many other countries, “biscuit” generally means a crisp, sweet baked treat—similar to a cookie.


In [5]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.edge.service import Service
from webdriver_manager.microsoft import EdgeChromiumDriverManager

from selenium.webdriver.common.by import By
from urllib.parse import quote
import time

import pandas as pd

In [6]:
file_path = "product_truth.xlsx"

dev_df = pd.read_excel(
    file_path,
    sheet_name="qa"
)

print(dev_df.shape)
print(dev_df.columns.tolist())

(412, 23)
['ITEM_CODE', 'NAN_KEY', 'EXTERNAL_CODE', 'COUNTRY', 'RETAILER_DESC', 'RETAILER', 'BRAND', 'PRODUCT_URL', 'REASONING', 'MODULE', 'GLOBAL_INTERSPACE_CLAIM', 'GLOBAL_CONSUMER_LIFESTAGE_CLAIM', 'GLOBAL_PACKAGING', 'GLOBAL_IF_MEDICATED', 'GLOBAL_PERCENTAGE_NATURAL_INGREDIENTS', 'GLOBAL_IF_WITH_SENSITIVE_CLAIM', 'GLOBAL_ORAL_CARE_FUNCTION', 'GLOBAL_IF_WITH_FLUORIDE', 'GLOBAL_FLAVOUR_FRAGRANCE_INGREDIENT_GROUP', 'GLOBAL_METHOD_OF_APPLICATION_DISPENSE', 'GLOBAL_PACKAGING_MATERIAL', 'GLOBAL_DESCRIPTIVE_SIZE_OF_TOOTHBRUSH_HEAD_CLAIM', 'GLOBAL_BRISTLE_STRENGTH_CLAIM']


In [7]:
dev_df.isnull().sum()

ITEM_CODE                                             0
NAN_KEY                                               0
EXTERNAL_CODE                                         0
COUNTRY                                               0
RETAILER_DESC                                         0
RETAILER                                              0
BRAND                                                 0
PRODUCT_URL                                         412
REASONING                                           412
MODULE                                              412
GLOBAL_INTERSPACE_CLAIM                             412
GLOBAL_CONSUMER_LIFESTAGE_CLAIM                     412
GLOBAL_PACKAGING                                    412
GLOBAL_IF_MEDICATED                                 412
GLOBAL_PERCENTAGE_NATURAL_INGREDIENTS               412
GLOBAL_IF_WITH_SENSITIVE_CLAIM                      412
GLOBAL_ORAL_CARE_FUNCTION                           412
GLOBAL_IF_WITH_FLUORIDE                         

In [10]:
dev_df[[
    "ITEM_CODE",
    "BRAND",
    "RETAILER_DESC",
    
]].head(20)

,ITEM_CODE,BRAND,RETAILER_DESC
0,515000000,BRILLIANT (LI & FUNG),brilliant teeth whitening 1 week charcoal kit ...
1,519000000,WISDOM (WISDOM TOOTHBRUSH),wiw tthwhtng stpchrcl 5s wisdom intense whiten...
2,55753033,POLIGRIP,**poli-grip liquid foam cleans125ml e0032
3,2130865,ORAL B,bcsan 20 x 1.7 gr (p) 20 e00d9
4,9764265,BINACA (HALEON),binaca pump breath fresh small p/m 25ml (enter...
5,4688813,PAN PARAG,pan parag pan masala tin 100g panparag pan mas...
6,509000000,ULTRADEX,ultradex one go unflavoured mouthwash on the g...
7,9315593,MINT ASURE,mnt assure frsh brth caps 75s*75 caps*sgl*std*...
8,7745480,HTC (HTC),handy tooth clean dental floss e00d9
9,546000000,PARLA (FORDEN),pärla pro high gloss whitening sensitive tooth...


In [ ]:
import json
import time
import pandas as pd

from azure.ai.inference.models import UserMessage


# =====================================================
# CONFIG
# =====================================================

MODEL_NAME = "hack-fest-gpt-5.6-luna"

SAVE_INTERVAL = 20

OUTPUT_FILE = "search_queries_qa.xlsx"

MAX_RETRIES = 5


# =====================================================
# PROMPT
# =====================================================

def build_prompt(row):

    return f"""
You are a product search query generator.

Create ONE short search query to find the exact product page.

Input:

Brand:
{row["BRAND"]}

Retailer Description:
{row["RETAILER_DESC"]}

Rules:

- Use only the minimum words needed.
- Use normal shopper search language.
- Include:
  Brand
  Variant
  Product type
  Size
  Pack count (if important)
  Flavor (if important)

- Remove:
  Internal codes
  Item codes
  SKU codes
  UNIT
  Retailer-only words
  Long numbers

- Expand obvious abbreviations:
  TP = toothpaste
  TB = toothbrush
  MW = mouthwash

Good examples:

Aquafresh whitening toothpaste pump 100ml

Colgate MaxFresh toothpaste 75g

Oral-B Pro Expert soft toothbrush 2 pack

Return ONLY JSON:

{{
  "search_query":""
}}
"""


# =====================================================
# MODEL CALL
# =====================================================

def generate_query(row):

    prompt = build_prompt(row)

    for attempt in range(MAX_RETRIES):

        try:

            response = client.complete(

                model=MODEL_NAME,

                messages=[
                    UserMessage(
                        content=prompt
                    )
                ],

                headers={
                    "Authorization": api_key
                }
            )

            raw = (
                response
                .choices[0]
                .message
                .content
            )

            result = json.loads(raw)

            return result.get(
                "search_query",
                ""
            )

        except Exception as ex:

            print(
                f"Retry {attempt+1}/{MAX_RETRIES}"
            )

            time.sleep(
                2 ** attempt
            )

    return ""


# =====================================================
# START FROM ROW 1
# =====================================================

dev_df["SEARCH_QUERY"] = ""

total_rows = len(dev_df)

print(
    f"Processing {total_rows} products"
)


# =====================================================
# GENERATE QUERY FOR ALL PRODUCTS
# =====================================================

for idx, row in dev_df.iterrows():

    try:

        search_query = generate_query(
            row
        )

        dev_df.loc[
            idx,
            "SEARCH_QUERY"
        ] = search_query

        print(
            f"{idx+1}/{total_rows}"
            f" -> "
            f"{search_query}"
        )

    except Exception as ex:

        print(
            f"Failed row {idx}"
        )

        print(ex)

        dev_df.loc[
            idx,
            "SEARCH_QUERY"
        ] = ""

    # -------------------------------------
    # SAVE EVERY 20 PRODUCTS
    # -------------------------------------

    if (

        (idx + 1)

        % SAVE_INTERVAL

        == 0

    ):

        dev_df.to_excel(
            OUTPUT_FILE,
            index=False
        )

        print(
            f"Checkpoint saved "
            f"at row {idx+1}"
        )


# =====================================================
# FINAL SAVE
# =====================================================

dev_df.to_excel(
    OUTPUT_FILE,
    index=False
)

print(
    "\nCompleted Successfully"
)

print(
    f"Saved: {OUTPUT_FILE}"
)
print(
    f"Total Products: {total_rows}"
)

Processing 412 products
1/412 -> Brilliant 1 week charcoal teeth whitening kit
2/412 -> WISDOM Intense Whitening Charcoal Strips 5 days
3/412 -> Poligrip liquid foaming cleanser 125ml
4/412 -> Oral-B Bcsan toothpaste 1.7g 20 pack
5/412 -> Binaca peppermint pump breath freshener 25ml
6/412 -> Pan Parag pan masala tin 100g
7/412 -> ULTRADEX One Go unflavoured mouthwash 10 sachets
8/412 -> MINT ASURE fresh breath capsules 75 count
9/412 -> HTC Handy Tooth Clean dental floss
10/412 -> PARLA Pro High Gloss Whitening Sensitive toothpaste tabs 62 count
11/412 -> SHALIMAR Excellent e cigarette
12/412 -> Colgate Sparkling Mint sugar-free dental gel toothpaste 18g 10 pack
13/412 -> DenTek natural floss picks 50 pack
14/412 -> Poligrip 3 Minute denture cleanser tablets 30 pack
15/412 -> PLACKERS
16/412 -> Aquafresh Milk Teeth toothbrush
17/412 -> BRUSHD Fresh Mint toothpaste tablets 125 pack
18/412 -> SETLERS Mintees mints 25g
19/412 -> Oral-B Shiny Clean 40 medium toothbrush 1 pack
20/412 -> Mor

In [13]:
import pandas as pd

# Load your file
df = pd.read_excel("search_queries_qa.xlsx")

# Remove SEARCH_QUERY from existing position
cols = list(df.columns)
cols.remove("SEARCH_QUERY")

# Find BRAND position
brand_idx = cols.index("BRAND")

# Insert SEARCH_QUERY after BRAND
cols.insert(
    brand_idx + 1,
    "SEARCH_QUERY"
)

# Reorder dataframe
df = df[cols]

# Save
df.to_excel(
    "search_queries_qa.xlsx",
    index=False
)

print("SEARCH_QUERY moved between BRAND and PRODUCT_URL")

SEARCH_QUERY moved between BRAND and PRODUCT_URL


In [14]:
d = pd.read_excel("search_queries_qa.xlsx")
d.shape

(412, 24)

In [ ]:
import json
import time
import pandas as pd

from urllib.parse import quote

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.edge.service import Service
from webdriver_manager.microsoft import EdgeChromiumDriverManager


# =====================================================
# CONFIG
# =====================================================

INPUT_FILE = "search_queries_qa.xlsx"

RAW_OUTPUT_FILE = "candidate_urls_raw_qa.json"

BATCH_SIZE = 3

SEARCH_RESULTS_LIMIT = 10

SEARCH_DELAY = 10

BATCH_COOLDOWN = 30


# =====================================================
# LOAD INPUT
# =====================================================

dev_df = pd.read_excel(INPUT_FILE)

print(f"Loaded {len(dev_df)} products")

print("Starting from Product 1")


# =====================================================
# STORAGE
# =====================================================

all_products_candidates = []


# =====================================================
# DRIVER
# =====================================================

options = webdriver.EdgeOptions()

options.add_argument("--disable-blink-features=AutomationControlled")

options.add_experimental_option(
    "excludeSwitches",
    ["enable-automation"]
)

options.add_experimental_option(
    "useAutomationExtension",
    False
)


driver = webdriver.Edge(
    service=Service(
        EdgeChromiumDriverManager().install()
    ),
    options=options
)


driver.execute_script(
    """
    Object.defineProperty(
        navigator,
        'webdriver',
        {
            get: () => undefined
        }
    )
    """
)



# =====================================================
# SAVE FUNCTION
# =====================================================

def save_checkpoint():

    with open(
        RAW_OUTPUT_FILE,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            all_products_candidates,
            f,
            indent=2,
            ensure_ascii=False
        )

    print(
        f"Checkpoint saved "
        f"({len(all_products_candidates)} products)"
    )


# =====================================================
# SEARCH FUNCTION
# =====================================================

def search_product(query):

    search_url = (
        "https://www.bing.com/search?q="
        + quote(str(query))
    )

    driver.get(search_url)

    time.sleep(5)


# =====================================================
# URL EXTRACTION
# =====================================================

def get_candidate_urls(max_results=10):

    urls = []

    results = driver.find_elements(
        By.CSS_SELECTOR,
        "li.b_algo h2 a"
    )

    for result in results[:max_results]:

        try:

            urls.append({
                "title": result.text,
                "url": result.get_attribute("href")
            })

        except Exception:
            pass

    return urls


# =====================================================
# CAPTCHA CHECK
# =====================================================

def verification_required():

    page = driver.page_source.lower()

    checks = [

        "verify you are human",

        "one last step",

        "solve the challenge",

        "challenge below"

    ]

    return any(
        check in page
        for check in checks
    )


# =====================================================
# WAIT FOR RESULTS
# =====================================================

def wait_for_results():

    for _ in range(60):

        time.sleep(2)

        page = driver.page_source.lower()

        links = driver.find_elements(
            By.CSS_SELECTOR,
            "li.b_algo h2 a"
        )

        if (
            "verify you are human" not in page
            and
            "one last step" not in page
            and
            len(links) > 0
        ):
            return True

    return False


# =====================================================
# PROCESS PRODUCTS
# =====================================================

batch_counter = 0

for idx, row in dev_df.iterrows():

    item_code = str(row["ITEM_CODE"])

    query = str(row["SEARCH_QUERY"])

    print(
        f"\n{'=' * 60}"
    )

    print(
        f"Product {idx + 1} of {len(dev_df)}"
    )

    print(
        f"ITEM_CODE : {item_code}"
    )

    print(
        f"QUERY     : {query}"
    )

    try:

        search_product(query)

        # ----------------------------------
        # CAPTCHA HANDLING
        # ----------------------------------

        if verification_required():

            print(
                "\nBING VERIFICATION DETECTED"
            )

            print(
                "Please complete verification "
                "inside Edge."
            )

            input(
                "\nPress ENTER once search "
                "results are visible..."
            )

            if not wait_for_results():

                result = {

                    "ITEM_CODE":
                        row["ITEM_CODE"],

                    "BRAND":
                        row["BRAND"],

                    "RETAILER_DESC":
                        row["RETAILER_DESC"],

                    "SEARCH_QUERY":
                        query,

                    "candidate_urls":
                        [],

                    "error":
                        "Verification completed but results not loaded"

                }

                all_products_candidates.append(
                    result
                )

                save_checkpoint()

                continue

        # ----------------------------------
        # URL EXTRACTION
        # ----------------------------------

        candidate_urls = get_candidate_urls(
            SEARCH_RESULTS_LIMIT
        )

        print(
            f"URLs Found: "
            f"{len(candidate_urls)}"
        )

        result = {

            "ITEM_CODE":
                row["ITEM_CODE"],

            "BRAND":
                row["BRAND"],

            "RETAILER_DESC":
                row["RETAILER_DESC"],

            "SEARCH_QUERY":
                query,

            "candidate_urls":
                candidate_urls

        }

        all_products_candidates.append(
            result
        )

        # ----------------------------------
        # SAVE AFTER EVERY PRODUCT
        # ----------------------------------

        save_checkpoint()

        batch_counter += 1

        # ----------------------------------
        # EVERY 3 PRODUCTS
        # ----------------------------------

        if batch_counter == BATCH_SIZE:

            print(
                f"\nBatch of {BATCH_SIZE} completed"
            )

            print(
                f"Cooling down for "
                f"{BATCH_COOLDOWN} seconds..."
            )

            time.sleep(
                BATCH_COOLDOWN
            )

            batch_counter = 0

        else:

            time.sleep(
                SEARCH_DELAY
            )

    except Exception as ex:

        print(
            f"ERROR: {str(ex)}"
        )

        result = {

            "ITEM_CODE":
                row["ITEM_CODE"],

            "BRAND":
                row["BRAND"],

            "RETAILER_DESC":
                row["RETAILER_DESC"],

            "SEARCH_QUERY":
                query,

            "candidate_urls":
                [],

            "error":
                str(ex)

        }

        all_products_candidates.append(
            result
        )

        save_checkpoint()

        time.sleep(
            SEARCH_DELAY
        )


# =====================================================
# FINAL SAVE
# =====================================================

driver.quit()

save_checkpoint()

print(
    "\nProcessing completed."
)

print(
    f"Results saved to "
    f"{RAW_OUTPUT_FILE}"
)

Loaded 412 products
Starting from Product 1

Product 1 of 412
ITEM_CODE : 515000000
QUERY     : Brilliant 1 week charcoal teeth whitening kit
URLs Found: 10
Checkpoint saved (1 products)

Product 2 of 412
ITEM_CODE : 519000000
QUERY     : WISDOM Intense Whitening Charcoal Strips 5 days
URLs Found: 10
Checkpoint saved (2 products)

Product 3 of 412
ITEM_CODE : 55753033
QUERY     : Poligrip liquid foaming cleanser 125ml
URLs Found: 10
Checkpoint saved (3 products)

Batch of 3 completed
Cooling down for 30 seconds...

Product 4 of 412
ITEM_CODE : 2130865
QUERY     : Oral-B Bcsan toothpaste 1.7g 20 pack
URLs Found: 10
Checkpoint saved (4 products)

Product 5 of 412
ITEM_CODE : 9764265
QUERY     : Binaca peppermint pump breath freshener 25ml
URLs Found: 10
Checkpoint saved (5 products)

Product 6 of 412
ITEM_CODE : 4688813
QUERY     : Pan Parag pan masala tin 100g
URLs Found: 10
Checkpoint saved (6 products)

Batch of 3 completed
Cooling down for 30 seconds...

Product 7 of 412
ITEM_CODE : 

In [22]:
len(all_products_candidates)

412

In [23]:
import json
import re
import base64
from urllib.parse import urlparse, parse_qs


INPUT_FILE = "candidate_urls_raw_qa.json"
OUTPUT_FILE = "candidate_urls_clean_qa.json"


def extract_href(html_string):

    match = re.search(
        r'href="([^"]+)"',
        str(html_string)
    )

    if match:
        return match.group(1)

    return str(html_string)


def decode_bing_redirect(url):

    try:

        if "bing.com/ck/" not in url:
            return url

        parsed = urlparse(url)

        params = parse_qs(parsed.query)

        if "u" not in params:
            return url

        encoded = params["u"][0]

        # remove Bing prefix a1
        if encoded.startswith("a1"):
            encoded = encoded[2:]

        # add missing padding
        encoded += "=" * (
            (4 - len(encoded) % 4) % 4
        )

        decoded = base64.b64decode(
            encoded
        ).decode(
            "utf-8",
            errors="ignore"
        )

        if decoded.startswith("http"):
            return decoded

        return url

    except Exception:
        return url


with open(
    INPUT_FILE,
    "r",
    encoding="utf-8"
) as f:

    data = json.load(f)


fixed_count = 0
bing_count = 0


for product in data:

    for candidate in product.get(
        "candidate_urls",
        []
    ):

        old_url = candidate["url"]

        # extract href first
        url = extract_href(old_url)

        # decode Bing redirects if present
        clean_url = decode_bing_redirect(url)

        if clean_url != old_url:
            fixed_count += 1

        if "bing.com/ck/" in clean_url:
            bing_count += 1

        candidate["url"] = clean_url


with open(
    OUTPUT_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        data,
        f,
        indent=2,
        ensure_ascii=False
    )


print(f"URLs cleaned: {fixed_count}")
print(f"Remaining Bing URLs: {bing_count}")
print(f"Saved: {OUTPUT_FILE}")

URLs cleaned: 3792
Remaining Bing URLs: 314
Saved: candidate_urls_clean_qa.json


In [24]:
import json

with open(
    "candidate_urls_clean_qa.json",
    "r",
    encoding="utf-8"
) as f:
    data = json.load(f)

remaining = []

for product in data:

    for candidate in product.get(
        "candidate_urls",
        []
    ):

        url = candidate["url"]

        if "bing.com/ck/" in str(url):

            remaining.append(

                {
                    "title": candidate["title"],
                    "url": url
                }

            )

print(
    f"Remaining Bing URLs: {len(remaining)}"
)

for row in remaining[:20]:
    print("\nTITLE:")
    print(row["title"])
    print("\nURL:")
    print(row["url"])

Remaining Bing URLs: 314

TITLE:
Amazon.in: Poligrip

URL:
https://www.bing.com/ck/a?!&&p=789d7ffffaa7c431426a4daf4e92138ddcedf0688222ad5116e6f5bb9e367732JmltdHM9MTc4OTk0ODgwMA&ptn=3&ver=2&hsh=4&fclid=22d4743f-8ae9-6331-1f48-63e48b9862df&psq=Poligrip+liquid+foaming+cleanser+125ml&u=a1aHR0cHM6Ly93d3cuYW1hem9uLmluL3BvbGlncmlwL3M_az1wb2xpZ3JpcA&ntb=1

TITLE:
Amazon.com: Poligrip Cleaner

URL:
https://www.bing.com/ck/a?!&&p=93fe59b06e23a6ed8fa82464fc988a84a4e16433440b5d880a88cea83bd05260JmltdHM9MTc4OTk0ODgwMA&ptn=3&ver=2&hsh=4&fclid=22d4743f-8ae9-6331-1f48-63e48b9862df&psq=Poligrip+liquid+foaming+cleanser+125ml&u=a1aHR0cHM6Ly93d3cuYW1hem9uLmNvbS9wb2xpZ3JpcC1jbGVhbmVyL3M_az1wb2xpZ3JpcCtjbGVhbmVy&ntb=1

TITLE:
Amazon.in: Oral B Toothpaste

URL:
https://www.bing.com/ck/a?!&&p=58ec0ddaad8c3b053c22919371ff03bdb4189c9b950411488962cf21bee282a2JmltdHM9MTc4OTk0ODgwMA&ptn=3&ver=2&hsh=4&fclid=22d4743f-8ae9-6331-1f48-63e48b9862df&psq=Oral-B+Bcsan+toothpaste+1.7g+20+pack&u=a1aHR0cHM6Ly93d3cuYW1hem9uLml

In [25]:
import json

with open(
    "candidate_urls_clean_qa.json",
    "r",
    encoding="utf-8"
) as f:
    data = json.load(f)

total = 0

bing = 0

for product in data:

    for candidate in product["candidate_urls"]:

        total += 1

        if "bing.com" in str(
            candidate["url"]
        ):
            bing += 1

print(
    f"Total URLs: {total}"
)

print(
    f"Bing URLs: {bing}"
)

print(
    f"Clean URLs: {total-bing}"
)

print(
    f"Success Rate: "
    f"{(total-bing)/total*100:.2f}%"
)

Total URLs: 4106
Bing URLs: 314
Clean URLs: 3792
Success Rate: 92.35%


In [26]:
import json
from urllib.parse import urlparse


INPUT_FILE = "candidate_urls_clean_qa.json"
OUTPUT_FILE = "candidate_urls_product_only_qa.json"


def is_product_url(url):

    try:

        parsed = urlparse(str(url))

        path = parsed.path.lower()

        # -----------------------------------
        # Skip empty/home pages
        # -----------------------------------

        if path in [
            "",
            "/",
            "/home",
            "/index.html"
        ]:
            return False

        # -----------------------------------
        # Skip search/category pages
        # -----------------------------------

        category_words = [

            "category",
            "search",
            "collections",
            "offers",
            "deals",
            "/s?",
            "/s/",
            "/search",
            "/browse"

        ]

        if any(
            x in path
            for x in category_words
        ):
            return False

        # -----------------------------------
        # Amazon
        # -----------------------------------

        if "amazon." in parsed.netloc:

            if (
                "/dp/" not in path
                and
                "/gp/product/" not in path
            ):
                return False

        # -----------------------------------
        # Flipkart
        # -----------------------------------

        if "flipkart." in parsed.netloc:

            if "/p/" not in path:
                return False

        # -----------------------------------
        # Generic product hint
        # -----------------------------------

        if (
            not any(c.isdigit() for c in path)
            and
            "-" not in path
        ):
            return False

        return True

    except:

        return False


# =====================================
# LOAD JSON
# =====================================

with open(
    INPUT_FILE,
    "r",
    encoding="utf-8"
) as f:

    data = json.load(f)


total_urls = 0
product_urls = 0


# =====================================
# FILTER URLS
# =====================================

for product in data:

    filtered_candidates = []

    for candidate in product.get(
        "candidate_urls",
        []
    ):

        total_urls += 1

        url = str(candidate["url"])

        if is_product_url(url):

            filtered_candidates.append(
                candidate
            )

            product_urls += 1

    product["candidate_urls"] = filtered_candidates


# =====================================
# SAVE
# =====================================

with open(
    OUTPUT_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        data,
        f,
        indent=2,
        ensure_ascii=False
    )


print(
    f"Total URLs checked: "
    f"{total_urls}"
)

print(
    f"Product URLs retained: "
    f"{product_urls}"
)

print(
    f"URLs removed: "
    f"{total_urls - product_urls}"
)

print(
    f"Saved: {OUTPUT_FILE}"
)

Total URLs checked: 4106
Product URLs retained: 2884
URLs removed: 1222
Saved: candidate_urls_product_only_qa.json


In [27]:
import json
import re


INPUT_FILE = "candidate_urls_product_only_qa.json"

OUTPUT_FILE = "candidate_urls_top3_qa.json"


# =====================================================
# CLEAN TEXT
# =====================================================

def normalize(text):

    text = str(text).lower()

    text = re.sub(
        r'[^a-z0-9\s]',
        ' ',
        text
    )

    words = text.split()

    return words


# =====================================================
# SCORE URL AGAINST QUERY
# =====================================================

def score_candidate(
    query,
    title,
    url
):

    query_words = set(
        normalize(query)
    )

    title_words = set(
        normalize(title)
    )

    url_words = set(
        normalize(url)
    )

    # ----------------------------------
    # Exact word overlap
    # ----------------------------------

    title_matches = len(
        query_words.intersection(
            title_words
        )
    )

    url_matches = len(
        query_words.intersection(
            url_words
        )
    )

    # ----------------------------------
    # Weighted score
    # ----------------------------------

    score = (

        title_matches * 10

        +

        url_matches * 5

    )

    # ----------------------------------
    # Bonus when most query words match
    # ----------------------------------

    if len(query_words) > 0:

        match_pct = (

            title_matches /

            len(query_words)

        )

        score += int(
            match_pct * 100
        )

    return score


# =====================================================
# LOAD JSON
# =====================================================

with open(
    INPUT_FILE,
    "r",
    encoding="utf-8"
) as f:

    data = json.load(f)


# =====================================================
# TOP 3 EXTRACTION
# =====================================================

products_processed = 0

for product in data:

    query = product.get(
        "SEARCH_QUERY",
        ""
    )

    candidates = product.get(
        "candidate_urls",
        []
    )

    scored_candidates = []

    for candidate in candidates:

        score = score_candidate(

            query,

            candidate.get(
                "title",
                ""
            ),

            candidate.get(
                "url",
                ""
            )

        )

        candidate["match_score"] = score

        scored_candidates.append(
            candidate
        )

    scored_candidates.sort(

        key=lambda x:
        x["match_score"],

        reverse=True

    )

    product[
        "candidate_urls"
    ] = scored_candidates[:3]

    products_processed += 1


# =====================================================
# SAVE
# =====================================================

with open(
    OUTPUT_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(

        data,

        f,

        indent=2,

        ensure_ascii=False

    )


print(
    f"Products Processed: "
    f"{products_processed}"
)

print(
    f"Saved: "
    f"{OUTPUT_FILE}"
)

Products Processed: 412
Saved: candidate_urls_top3_qa.json


In [1]:
import json
import time
import math

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.edge.service import Service
from webdriver_manager.microsoft import EdgeChromiumDriverManager


# ====================================
# CONFIG
# ====================================

INPUT_FILE = "candidate_urls_top3_qa.json"

OUTPUT_PREFIX = "product_text_batch_qa"

SAVE_EVERY_PRODUCTS = 10


# ====================================
# LOAD JSON
# ====================================

with open(
    INPUT_FILE,
    "r",
    encoding="utf-8"
) as f:

    data = json.load(f)

TOTAL_PRODUCTS = len(data)

print(
    f"Loaded {TOTAL_PRODUCTS} products"
)


# ====================================
# DRIVER
# ====================================

options = webdriver.EdgeOptions()

options.add_argument("--headless=new")

driver = webdriver.Edge(

    service=Service(
        EdgeChromiumDriverManager().install()
    ),

    options=options

)


# ====================================
# TEXT EXTRACTION
# ====================================

def extract_product_text(url):

    try:

        driver.get(url)

        time.sleep(5)

        body_text = driver.find_element(
            By.TAG_NAME,
            "body"
        ).text

        title = driver.title

        return {

            "page_title":
                title,

            "page_text":
                body_text[:15000],

            "text_length":
                len(body_text)

        }

    except Exception as ex:

        return {

            "page_title":
                "",

            "page_text":
                "",

            "text_length":
                0,

            "error":
                str(ex)

        }


# ====================================
# SAVE BATCH
# ====================================

def save_batch(
    batch_data,
    batch_number
):

    filename = (

        f"{OUTPUT_PREFIX}_"

        f"{batch_number:03d}.json"

    )

    with open(

        filename,

        "w",

        encoding="utf-8"

    ) as f:

        json.dump(

            batch_data,

            f,

            indent=2,

            ensure_ascii=False

        )

    print(
        f"\n✅ Saved {filename}"
    )


# ====================================
# PROCESS
# ====================================

products_processed = 0

url_processed = 0

successful_urls = 0

batch_number = 1

current_batch = []


for product_index, product in enumerate(
    data,
    start=1
):

    print(
        "\n"
        + "=" * 100
    )

    print(
        f"PRODUCT "
        f"{product_index}"
        f" OF "
        f"{TOTAL_PRODUCTS}"
    )

    print(
        f"ITEM_CODE : "
        f"{product['ITEM_CODE']}"
    )

    candidates = product.get(
        "candidate_urls",
        []
    )

    for url_index, candidate in enumerate(
        candidates,
        start=1
    ):

        url_processed += 1

        url = candidate["url"]

        print(
            f"\nURL "
            f"{url_index}"
            f"/"
            f"{len(candidates)}"
        )

        print(
            f"Scraping: {url}"
        )

        page_data = extract_product_text(
            url
        )

        candidate.update(
            page_data
        )

        if page_data[
            "text_length"
        ] > 0:

            successful_urls += 1

            print(
                f"✅ "
                f"{page_data['text_length']:,}"
                f" chars"
            )

        else:

            print(
                "❌ No text"
            )

    current_batch.append(
        product
    )

    products_processed += 1

    # ---------------------------
    # SAVE EVERY 10 PRODUCTS
    #---------------------------

    if (

        products_processed
        %
        SAVE_EVERY_PRODUCTS
        ==
        0

    ):

        save_batch(

            current_batch,

            batch_number

        )

        batch_number += 1

        current_batch = []


# ====================================
# SAVE LAST PARTIAL BATCH
# ====================================

if len(current_batch) > 0:

    save_batch(

        current_batch,

        batch_number

    )


# ====================================
# CLEANUP
# ====================================

driver.quit()


print(
    "\n"
    + "=" * 100
)

print(
    "PROCESSING COMPLETE"
)

print(
    f"Products Processed : "
    f"{products_processed}"
)

print(
    f"URLs Processed : "
    f"{url_processed}"
)

print(
    f"Successful URLs : "
    f"{successful_urls}"
)

print(
    f"Batches Created : "
    f"{batch_number}"
)

Loaded 412 products

PRODUCT 1 OF 412
ITEM_CODE : 515000000

URL 1/3
Scraping: https://www.superdrug.com/toiletries/dental/whitening-kits/brilliant-whitening-1-week-charcoal-kit/p/773567
✅ 274 chars

URL 2/3
Scraping: https://www.ebay.com/p/10037910285?msockid=22d4743f8ae963311f4863e48b9862df
✅ 4,257 chars

URL 3/3
Scraping: https://whatbritainbuys.com/products/brilliant-1-week-tooth-whitening-kit-4-shades-whiter-in-1-week-prevents-stains-re-occurring-controls-tartar-clinically-proven
✅ 335 chars

PRODUCT 2 OF 412
ITEM_CODE : 519000000

URL 1/3
Scraping: https://www.youtube.com/shorts/5ot85VpzFNE&ntb=1?msockid=ed5f3a15b58c11f1a0c0f0b7d328162a
❌ No text

URL 2/3
Scraping: https://fineessentialoils.com/oral-care/best-charcoal-teeth-whitening-kit/
✅ 17,013 chars

URL 3/3
Scraping: https://www.dentalroundup.com/best/best-whitening-strips-sensitive-teeth/
✅ 26,941 chars

PRODUCT 3 OF 412
ITEM_CODE : 55753033

URL 1/3
Scraping: https://nakosite.com/product/poligrip-denture-freshfoam-liquid-c

In [2]:
import json
import glob
import os

# =====================================
# CONFIG
# =====================================

INPUT_PATTERN = "product_text_batch_qa_*.json"

OUTPUT_FILE = "text_evidence_dev_qa.json"

# =====================================
# GET ALL BATCH FILES
# =====================================

batch_files = sorted(
    glob.glob(INPUT_PATTERN)
)

print(
    f"Found {len(batch_files)} batch files"
)

# =====================================
# MERGE
# =====================================

all_products = []

for file in batch_files:

    print(
        f"Reading: {file}"
    )

    try:

        with open(
            file,
            "r",
            encoding="utf-8"
        ) as f:

            batch_data = json.load(f)

        print(
            f"Products in file: "
            f"{len(batch_data)}"
        )

        all_products.extend(
            batch_data
        )

    except Exception as ex:

        print(
            f"ERROR reading {file}"
        )

        print(ex)

# =====================================
# SAVE MASTER JSON
# =====================================

with open(
    OUTPUT_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        all_products,
        f,
        indent=2,
        ensure_ascii=False
    )

print(
    "\n"
    + "=" * 80
)

print(
    f"Total Products Merged: "
    f"{len(all_products)}"
)

print(
    f"Saved Master File: "
    f"{OUTPUT_FILE}"
)

Found 42 batch files
Reading: product_text_batch_qa_001.json
Products in file: 10
Reading: product_text_batch_qa_002.json
Products in file: 10
Reading: product_text_batch_qa_003.json
Products in file: 10
Reading: product_text_batch_qa_004.json
Products in file: 10
Reading: product_text_batch_qa_005.json
Products in file: 10
Reading: product_text_batch_qa_006.json
Products in file: 10
Reading: product_text_batch_qa_007.json
Products in file: 10
Reading: product_text_batch_qa_008.json
Products in file: 10
Reading: product_text_batch_qa_009.json
Products in file: 10
Reading: product_text_batch_qa_010.json
Products in file: 10
Reading: product_text_batch_qa_011.json
Products in file: 10
Reading: product_text_batch_qa_012.json
Products in file: 10
Reading: product_text_batch_qa_013.json
Products in file: 10
Reading: product_text_batch_qa_014.json
Products in file: 10
Reading: product_text_batch_qa_015.json
Products in file: 10
Reading: product_text_batch_qa_016.json
Products in file: 10
Rea

In [8]:
import json


INPUT_FILE = "text_evidence_dev_qa.json"

OUTPUT_FILE = "cleaned_text_evidence_dev_qa.json"


# ==========================================
# LOAD
# ==========================================

with open(
    INPUT_FILE,
    "r",
    encoding="utf-8"
) as f:

    data = json.load(f)


# ==========================================
# PROCESS
# ==========================================

products_processed = 0

urls_processed = 0


for product_index, product in enumerate(
    data,
    start=1
):

    print(
        "\n"
        + "=" * 100
    )

    print(
        f"PRODUCT "
        f"{product_index}"
        f" OF "
        f"{len(data)}"
    )

    print(
        f"ITEM_CODE : "
        f"{product['ITEM_CODE']}"
    )

    candidates = product.get(
        "candidate_urls",
        []
    )

    for url_index, candidate in enumerate(
        candidates,
        start=1
    ):

        urls_processed += 1

        page_text = candidate.get(
            "page_text",
            ""
        )

        print(
            f"\nURL {url_index}"
        )

        if not page_text:

            candidate[
                "clean_text"
            ] = ""

            continue

        prompt = f"""
You are extracting product information from an ecommerce product page.

Remove:
- Navigation
- Search controls
- Cart
- Login/account sections
- Delivery/location sections
- Similar products
- Recommended products
- Footer content
- Privacy/legal text
- Advertisements

Keep ONLY:

- Product name
- Brand
- Variant
- Pack size
- Claims
- Ingredients
- Features
- Packaging
- Product specifications
- GTIN / UPC / EAN
- Product identifiers

Raw Text:

{page_text}

Return only cleaned product information.
"""

        try:

            clean_text = ask_llm(
                prompt
            )

        except Exception as ex:

            clean_text = (
                f"ERROR: {ex}"
            )

        candidate[
            "clean_text"
        ] = clean_text

        print(
            clean_text[:1000]
        )

    products_processed += 1

    # ----------------------------------
    # SAVE EVERY 10 PRODUCTS
    # ----------------------------------

    if products_processed % 10 == 0:

        with open(
            OUTPUT_FILE,
            "w",
            encoding="utf-8"
        ) as f:

            json.dump(
                data,
                f,
                indent=2,
                ensure_ascii=False
            )

        print(
            f"\nCheckpoint saved "
            f"after "
            f"{products_processed} products"
        )


# ==========================================
# FINAL SAVE
# ==========================================

with open(
    OUTPUT_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        data,
        f,
        indent=2,
        ensure_ascii=False
    )

print(
    "\n"
    + "=" * 100
)

print(
    f"Products Processed: "
    f"{products_processed}"
)

print(
    f"URLs Processed: "
    f"{urls_processed}"
)

print(
    f"Saved: "
    f"{OUTPUT_FILE}"
)


PRODUCT 1 OF 412
ITEM_CODE : 515000000

URL 1
Product name: Brilliant Whitening 1 Week Charcoal Kit  
Brand: Brilliant

URL 2
Product name: Brilliant 1 Week Teeth Whitening Charcoal Kit  
Brand: Brilliant  
Variant: Charcoal  
Pack size: X2  
Claims: 4 shades whiter in one week*  
MPN: XCF34  
GTIN: 5014697056627  
UPC: 5014697056627  
eBay Product ID (ePID): 10037910285

URL 3
No product information available.

PRODUCT 2 OF 412
ITEM_CODE : 519000000

URL 1

URL 2
## GuruNanda Total Smile Makeover Oral Care Kit, Teeth

- **Brand:** GuruNanda
- **Variant:** Total Smile Makeover Oral Care Kit
- **Product type:** Teeth whitening and oral care kit
- **Claims:**
  - Gentle for sensitive teeth and gums
  - Supports gum health and fresh breath
  - Ayurvedic-inspired oral care
  - Travel-friendly
  - Helps remove stains and whiten teeth
  - Fluoride-free mouthwash
- **Ingredients:**
  - Hydrogen peroxide
  - Fractionated coconut oil
  - Coconut extracts
  - Seven essential oils
  - Vitamins D

In [9]:
import json

INPUT_FILE = "cleaned_text_evidence_dev_qa.json"

OUTPUT_FILE = "final_cleaned_text_evidence_dev_qa.json"


# ==================================
# LOAD JSON
# ==================================

with open(
    INPUT_FILE,
    "r",
    encoding="utf-8"
) as f:

    data = json.load(f)


# ==================================
# REMOVE UNWANTED FIELDS
# ==================================

for product in data:

    candidates = product.get(
        "candidate_urls",
        []
    )

    for candidate in candidates:

        candidate.pop(
            "match_score",
            None
        )

        candidate.pop(
            "page_text",
            None
        )

        candidate.pop(
            "text_length",
            None
        )


# ==================================
# SAVE
# ==================================

with open(
    OUTPUT_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        data,
        f,
        indent=2,
        ensure_ascii=False
    )

print(
    f"Saved cleaned file: "
    f"{OUTPUT_FILE}"
)

print(
    f"Products: "
    f"{len(data)}"
)

Saved cleaned file: final_cleaned_text_evidence_dev_qa.json
Products: 412


In [10]:
import json
import re
import time
import requests

from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.edge.options import Options
from selenium.webdriver.edge.service import Service

from webdriver_manager.microsoft import (
    EdgeChromiumDriverManager
)

# =====================================================
# CONFIG
# =====================================================

INPUT_FILE = "final_cleaned_text_evidence_dev_qa.json"

OUTPUT_FILE = "cleaned_text_evidence_with_images_qa.json"

SAVE_EVERY_PRODUCTS = 10

# =====================================================
# HEADERS
# =====================================================

HEADERS = {

    "User-Agent": (
        "Mozilla/5.0 "
        "(Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 "
        "(KHTML, like Gecko) "
        "Chrome/116.0 Safari/537.36"
    ),

    "Accept-Language":
        "en-US,en;q=0.9"

}

# =====================================================
# EXTRACT REAL URL
# =====================================================

def clean_url(raw_url):

    raw_url = str(raw_url)

    match = re.search(
        r'href="([^"]+)"',
        raw_url
    )

    if match:
        return match.group(1)

    return raw_url

# =====================================================
# EDGE DRIVER
# =====================================================

options = Options()

options.add_argument("--headless")

options.add_argument("--disable-gpu")

options.add_argument("--no-sandbox")

driver = webdriver.Edge(
    service=Service(
        EdgeChromiumDriverManager().install()
    ),
    options=options
)

# =====================================================
# LOAD PAGE WITH EDGE
# =====================================================

def get_html_with_edge(url):

    try:

        driver.get(url)

        time.sleep(3)

        return driver.page_source

    except:

        return ""

# =====================================================
# IMAGE URL IMPROVEMENT
# =====================================================

def upgrade_image_url(
    image_url,
    page_url
):

    if not image_url:
        return ""

    try:

        # Amazon HD image

        if "amazon" in page_url.lower():

            image_url = re.sub(
                r"\._.*?\_\.jpg",
                ".jpg",
                image_url
            )

        # Flipkart HD image

        if "flipkart" in page_url.lower():

            image_url = re.sub(
                r"/\d+/\d+/",
                "/1080/1080/",
                image_url
            )

    except:
        pass

    return image_url



# =====================================================
# DETECT HTML TYPE
# =====================================================

def detect_image_loading(url):

    try:

        html = requests.get(
            url,
            headers=HEADERS,
            timeout=10
        ).text

    except:

        html = get_html_with_edge(url)

    soup = BeautifulSoup(
        html,
        "html.parser"
    )

    meta_img = (

        soup.find(
            "meta",
            property="og:image"
        )

        or

        soup.find(
            "meta",
            attrs={
                "name":
                "twitter:image"
            }
        )

    )

    if (
        meta_img
        and
        meta_img.get("content")
    ):
        return "meta"

    if soup.find("img"):
        return "static"

    return "dynamic"

# =====================================================
# META IMAGE
# =====================================================

def get_images_meta(url):

    try:

        html = requests.get(
            url,
            headers=HEADERS,
            timeout=10
        ).text

    except:

        html = get_html_with_edge(url)

    soup = BeautifulSoup(
        html,
        "html.parser"
    )

    meta_img = (

        soup.find(
            "meta",
            property="og:image"
        )

        or

        soup.find(
            "meta",
            attrs={
                "name":
                "twitter:image"
            }
        )

    )

    if (
        meta_img
        and
        meta_img.get("content")
    ):

        return [

            upgrade_image_url(
                meta_img["content"],
                url
            )

        ]

    return []

# =====================================================
# STATIC IMAGES
# =====================================================

def get_images_static(url):

    try:

        html = requests.get(
            url,
            headers=HEADERS,
            timeout=10
        ).text

    except:

        html = get_html_with_edge(url)

    soup = BeautifulSoup(
        html,
        "html.parser"
    )

    images = []

    for img in soup.find_all("img"):

        src = img.get("src")

        if not src:
            continue

        if not src.startswith("http"):
            continue

        images.append(

            upgrade_image_url(
                src,
                url
            )

        )

    return images

# =====================================================
# DYNAMIC IMAGES
# =====================================================

def get_images_dynamic(url):

    html = get_html_with_edge(url)

    soup = BeautifulSoup(
        html,
        "html.parser"
    )

    images = []

    for img in soup.find_all("img"):

        src = img.get("src")

        if not src:
            continue

        if not src.startswith("http"):
            continue

        images.append(

            upgrade_image_url(
                src,
                url
            )

        )

    return images

# =====================================================
# PRODUCT IMAGE FUNCTION
# =====================================================

def get_product_images(url):

    method = detect_image_loading(
        url
    )

    if method == "meta":

        return get_images_meta(
            url
        )

    elif method == "static":

        return get_images_static(
            url
        )

    else:

        return get_images_dynamic(
            url
        )

# =====================================================
# LOAD JSON
# =====================================================

with open(
    INPUT_FILE,
    "r",
    encoding="utf-8"
) as f:

    data = json.load(f)

TOTAL_PRODUCTS = len(data)

print(
    f"Loaded {TOTAL_PRODUCTS} products"
)

# =====================================================
# PROCESS
# =====================================================

images_found = 0

products_processed = 0

for product_index, product in enumerate(
    data,
    start=1
):

    print(
        "\n" + "=" * 100
    )

    print(
        f"PRODUCT {product_index}/{TOTAL_PRODUCTS}"
    )

    print(
        f"ITEM_CODE : {product['ITEM_CODE']}"
    )

    candidates = product.get(
        "candidate_urls",
        []
    )

    for url_index, candidate in enumerate(
        candidates,
        start=1
    ):

        original_url = candidate.get(
            "url",
            ""
        )

        real_url = clean_url(
            original_url
        )

        print(
            f"\nURL {url_index}/{len(candidates)}"
        )

        print("PAGE URL:")

        print(real_url)

        images = get_product_images(
            real_url
        )

        if len(images) > 0:

            candidate[
                "image_url"
            ] = images[0]

            images_found += 1

            print(
                "\nIMAGE URL:"
            )

            print(
                images[0]
            )

        else:

            candidate[
                "image_url"
            ] = ""

            print(
                "\nNO IMAGE FOUND"
            )

        print(
            "-" * 80
        )

    products_processed += 1

    # =====================================
    # SAVE EVERY 10 PRODUCTS
    # =====================================

    if (

        products_processed
        %
        SAVE_EVERY_PRODUCTS
        ==
        0

    ):

        with open(
            OUTPUT_FILE,
            "w",
            encoding="utf-8"
        ) as f:

            json.dump(
                data,
                f,
                indent=2,
                ensure_ascii=False
            )

        print(
            f"\n✅ CHECKPOINT SAVED"
        )

        print(
            f"Products Processed: "
            f"{products_processed}"
        )

# =====================================================
# SAVE FINAL
# =====================================================

driver.quit()

with open(
    OUTPUT_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        data,
        f,
        indent=2,
        ensure_ascii=False
    )

print(
    "\n" + "=" * 100
)

print(
    f"Products Processed: {products_processed}"
)

print(
    f"Images Found: {images_found}"
)

print(
    f"Saved: {OUTPUT_FILE}"
)

Loaded 412 products

PRODUCT 1/412
ITEM_CODE : 515000000

URL 1/3
PAGE URL:
https://www.superdrug.com/toiletries/dental/whitening-kits/brilliant-whitening-1-week-charcoal-kit/p/773567

NO IMAGE FOUND
--------------------------------------------------------------------------------

URL 2/3
PAGE URL:
https://www.ebay.com/p/10037910285?msockid=22d4743f8ae963311f4863e48b9862df

IMAGE URL:
https://i.ebayimg.com/images/g/7WgAAOSwQ7haxTU1/s-l1600.jpg
--------------------------------------------------------------------------------

URL 3/3
PAGE URL:
https://whatbritainbuys.com/products/brilliant-1-week-tooth-whitening-kit-4-shades-whiter-in-1-week-prevents-stains-re-occurring-controls-tartar-clinically-proven

IMAGE URL:
http://whatbritainbuys.com/cdn/shop/products/414IOPJkUCL_e710bcb6-0260-483b-8613-70b0bb212260_grande.jpg?v=1596692840
--------------------------------------------------------------------------------

PRODUCT 2/412
ITEM_CODE : 519000000

URL 1/3
PAGE URL:
https://www.youtube.co

In [11]:
import json
import re
import os

from azure.ai.inference.models import (
    UserMessage
)

# =====================================================
# CONFIG
# =====================================================

INPUT_FILE = (
    "cleaned_text_evidence_with_images_qa.json"
)

OUTPUT_FILE = (
    "ocr_verified_products_qa.json"
)

CHECKPOINT_EVERY = 10

# Resume support
START_FROM_PRODUCT = 1


# =====================================================
# HELPER
# =====================================================

def clean_url(value):

    value = str(value)

    href_match = re.search(
        r'href="([^"]+)"',
        value
    )

    if href_match:
        return href_match.group(1)

    url_match = re.search(
        r'https?://[^\s<>"\']+',
        value
    )

    if url_match:
        return url_match.group(0)

    return value


def normalize(text):

    text = str(text).lower()

    text = re.sub(
        r'[^a-z0-9 ]',
        ' ',
        text
    )

    return text


# =====================================================
# MATCH SCORING
# =====================================================

def calculate_match_score(
    product,
    ocr_output
):

    score = 0

    brand_expected = normalize(
        product.get(
            "BRAND",
            ""
        )
    )

    query_expected = normalize(
        product.get(
            "SEARCH_QUERY",
            ""
        )
    )

    ocr_text = normalize(

        str(
            ocr_output.get(
                "brand",
                ""
            )
        )

        + " "

        + str(
            ocr_output.get(
                "product_name",
                ""
            )
        )

        + " "

        + str(
            ocr_output.get(
                "variant",
                ""
            )
        )

        + " "

        + str(
            ocr_output.get(
                "pack_size",
                ""
            )
        )

        + " "

        + " ".join(
            ocr_output.get(
                "visible_pack_text",
                []
            )
        )

    )

    # ------------------------
    # Brand Match
    # ------------------------

    for word in brand_expected.split():

        if (

            len(word) > 2

            and

            word in ocr_text

        ):

            score += 40
            break

    # ------------------------
    # Search Query Match
    # ------------------------

    query_words = set(
        query_expected.split()
    )

    matches = 0

    for word in query_words:

        if len(word) < 3:
            continue

        if word in ocr_text:

            matches += 1

    score += (
        matches * 10
    )

    # ------------------------
    # Product Pack Detected
    # ------------------------

    if ocr_output.get(
        "image_contains_product_pack"
    ):

        score += 20

    return min(
        score,
        100
    )


# =====================================================
# LOAD JSON
# =====================================================

with open(
    INPUT_FILE,
    "r",
    encoding="utf-8"
) as f:

    data = json.load(f)

TOTAL_PRODUCTS = len(data)

print(
    f"Loaded {TOTAL_PRODUCTS} products"
)

products_processed = 0

images_processed = 0


# =====================================================
# MAIN LOOP
# =====================================================

for product_index, product in enumerate(
    data,
    start=1
):

    if (
        product_index
        <
        START_FROM_PRODUCT
    ):
        continue

    print(
        "\n"
        + "=" * 100
    )

    print(
        f"PRODUCT "
        f"{product_index}"
        f"/"
        f"{TOTAL_PRODUCTS}"
    )

    print(
        f"ITEM_CODE : "
        f"{product['ITEM_CODE']}"
    )

    best_score = -1

    best_candidate = None

    candidates = product.get(
        "candidate_urls",
        []
    )

    for candidate_index, candidate in enumerate(
        candidates,
        start=1
    ):

        image_url = clean_url(
            candidate.get(
                "image_url",
                ""
            )
        )

        if not image_url:
            continue

        print(
            f"\nIMAGE "
            f"{candidate_index}"
        )

        print(
            image_url
        )

        prompt = f"""
You are performing OCR and PRODUCT IMAGE VERIFICATION.

Expected Product Information

BRAND:
{product.get("BRAND","")}

SEARCH QUERY:
{product.get("SEARCH_QUERY","")}

RETAILER DESCRIPTION:
{product.get("RETAILER_DESC","")}

PAGE TITLE:
{candidate.get("page_title","")}

PAGE PRODUCT TEXT:
{candidate.get("clean_text","")}

Rules:

1. Use ONLY visible image content.

2. Ignore:
   - image URL
   - filename
   - product page text

3. If image is:
   logo,
   icon,
   retailer banner,
   navigation image,
   unrelated image

Return:
image_contains_product_pack=false

4. Compare image content with expected product.

5. If image matches:
image_relevant=true

6. If image is different:
image_relevant=false

Return JSON ONLY

{{
  "image_contains_product_pack": true,
  "image_relevant": true,

  "brand": "",
  "product_name": "",
  "variant": "",
  "pack_size": "",

  "claims": [],
  "packaging": "",

  "visible_pack_text": [],

  "image_evidence_count": 0
}}
"""

        try:

            response = client.complete(

                model=
                "hack-fest-gpt-5.6-luna",

                messages=[

                    UserMessage(

                        content=[

                            {
                                "type":
                                "text",

                                "text":
                                prompt
                            },

                            {
                                "type":
                                "image_url",

                                "image_url":
                                {
                                    "url":
                                    image_url
                                }
                            }

                        ]

                    )

                ],

                headers={
                    "Authorization":
                    api_key
                }

            )

            raw_response = (

                response
                .choices[0]
                .message
                .content

            )

            try:

                ocr = json.loads(
                    raw_response
                )

            except Exception:

                ocr = {

                    "parse_error":
                    True,

                    "raw_response":
                    raw_response

                }

            score = calculate_match_score(
                product,
                ocr
            )

            candidate[
                "ocr"
            ] = ocr

            candidate[
                "image_match_score"
            ] = score

            candidate[
                "image_relevant"
            ] = (
                score >= 50
            )

            images_processed += 1

            print(
                f"SCORE: {score}"
            )

            if score > best_score:

                best_score = score

                best_candidate = candidate

        except Exception as ex:

            candidate[
                "ocr"
            ] = {

                "error":
                str(ex)

            }

            candidate[
                "image_match_score"
            ] = 0

            candidate[
                "image_relevant"
            ] = False

            print(ex)

    # =================================================
    # BEST IMAGE EVIDENCE
    # =================================================

    product[
        "best_image_evidence"
    ] = best_candidate

    product[
        "best_image_score"
    ] = best_score

    products_processed += 1

    # =================================================
    # CHECKPOINT
    # =================================================

    if (

        products_processed
        %
        CHECKPOINT_EVERY
        ==
        0

    ):

        with open(
            OUTPUT_FILE,
            "w",
            encoding="utf-8"
        ) as f:

            json.dump(
                data,
                f,
                indent=2,
                ensure_ascii=False
            )

        print(
            "\nCHECKPOINT SAVED"
        )

        print(
            f"Products Processed: "
            f"{products_processed}"
        )

# =====================================================
# FINAL SAVE
# =====================================================

with open(
    OUTPUT_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        data,
        f,
        indent=2,
        ensure_ascii=False
    )

print(
    "\n"
    + "=" * 100
)

print(
    f"Products Processed: "
    f"{products_processed}"
)

print(
    f"Images Processed: "
    f"{images_processed}"
)

print(
    f"Saved: "
    f"{OUTPUT_FILE}"
)

Loaded 412 products

PRODUCT 1/412
ITEM_CODE : 515000000

IMAGE 2
https://i.ebayimg.com/images/g/7WgAAOSwQ7haxTU1/s-l1600.jpg
SCORE: 0

IMAGE 3
http://whatbritainbuys.com/cdn/shop/products/414IOPJkUCL_e710bcb6-0260-483b-8613-70b0bb212260_grande.jpg?v=1596692840
SCORE: 100

PRODUCT 2/412
ITEM_CODE : 519000000

IMAGE 2
https://fineessentialoils.com/wp-content/uploads/2021/12/cropped-Lucid_Origin_Create_a_sophisticated_professional_logo_for_Fine_0-removebg-preview.png
SCORE: 0

IMAGE 3
https://www.dentalroundup.com/og-default.png
(400) litellm.BadRequestError: AzureException BadRequestError - Failed to download image from https://www.dentalroundup.com/og-default.png.No fallback model group found for original model_group=hack-fest-gpt-5.6-luna. Fallbacks=[{'fallback-test-gpt-4.1': ['fallback-test-gpt-4.1-common', 'fallback-test-gpt-4.1-paygo']}]. Received Model Group=hack-fest-gpt-5.6-luna
Available Model Group Fallbacks=None
Error doing the fallback: litellm.BadRequestError: AzureExceptio

In [12]:
import json

INPUT_FILE = "ocr_verified_products_qa.json"

OUTPUT_FILE = (
    "ocr_verified_products_filtered_qa.json"
)

REMOVED_FILE = (
    "ocr_verified_products_removed_qa.json"
)

# =====================================================
# LOAD
# =====================================================

with open(
    INPUT_FILE,
    "r",
    encoding="utf-8"
) as f:

    data = json.load(f)

products_processed = 0

candidates_removed = 0

candidates_kept = 0

removed_products = []


# =====================================================
# FILTER
# =====================================================

for product in data:

    filtered_candidates = []

    removed_candidates = []

    for candidate in product.get(
        "candidate_urls",
        []
    ):

        # ---------------------------------
        # Keep candidates with NO score
        # ---------------------------------

        if (
            "image_match_score"
            not in candidate
        ):

            filtered_candidates.append(
                candidate
            )

            candidates_kept += 1

            continue

        score = candidate.get(
            "image_match_score"
        )

        # ---------------------------------
        # Keep NULL score
        # ---------------------------------

        if score is None:

            filtered_candidates.append(
                candidate
            )

            candidates_kept += 1

            continue

        # ---------------------------------
        # Remove score <= 25
        # ---------------------------------

        if score <= 25:

            removed_candidates.append(
                candidate
            )

            candidates_removed += 1

            continue

        filtered_candidates.append(
            candidate
        )

        candidates_kept += 1

    # ---------------------------------
    # Main file
    # ---------------------------------

    product[
        "candidate_urls"
    ] = filtered_candidates

    # ---------------------------------
    # Removed file
    # ---------------------------------

    if len(removed_candidates):

        removed_products.append({

            "ITEM_CODE":
                product.get(
                    "ITEM_CODE"
                ),

            "BRAND":
                product.get(
                    "BRAND"
                ),

            "SEARCH_QUERY":
                product.get(
                    "SEARCH_QUERY"
                ),

            "removed_candidates":
                removed_candidates

        })

    # ---------------------------------
    # Best Evidence Recalculation
    # ---------------------------------

    scored_candidates = [

        c

        for c in filtered_candidates

        if (
            "image_match_score"
            in c
            and
            c["image_match_score"]
            is not None
        )

    ]

    if len(scored_candidates):

        best_candidate = max(

            scored_candidates,

            key=lambda x:
            x.get(
                "image_match_score",
                0
            )

        )

        product[
            "best_image_evidence"
        ] = best_candidate

        product[
            "best_image_score"
        ] = best_candidate.get(
            "image_match_score",
            0
        )

    products_processed += 1


# =====================================================
# SAVE FILTERED
# =====================================================

with open(
    OUTPUT_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        data,
        f,
        indent=2,
        ensure_ascii=False
    )

# =====================================================
# SAVE REMOVED
# =====================================================

with open(
    REMOVED_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        removed_products,
        f,
        indent=2,
        ensure_ascii=False
    )

# =====================================================
# SUMMARY
# =====================================================

print(
    "\n========================================"
)

print(
    f"Products Processed : "
    f"{products_processed}"
)

print(
    f"Candidates Kept : "
    f"{candidates_kept}"
)

print(
    f"Candidates Removed : "
    f"{candidates_removed}"
)

print(
    f"Filtered File : "
    f"{OUTPUT_FILE}"
)

print(
    f"Removed File : "
    f"{REMOVED_FILE}"
)


Products Processed : 412
Candidates Kept : 699
Candidates Removed : 512
Filtered File : ocr_verified_products_filtered_qa.json
Removed File : ocr_verified_products_removed_qa.json


In [13]:
import json
import re

from azure.ai.inference.models import (
    UserMessage
)

# =====================================================
# CONFIG
# =====================================================

INPUT_FILE = (
    "ocr_verified_products_filtered_qa.json"
)

OUTPUT_FILE = (
    "final_product_truth_qa.json"
)

CHECKPOINT_EVERY = 10

START_FROM_PRODUCT = 1

# =====================================================
# URL CLEANER
# =====================================================

def clean_url(value):

    value = str(value)

    href_match = re.search(
        r'href="([^"]+)"',
        value
    )

    if href_match:
        return href_match.group(1)

    url_match = re.search(
        r'https?://[^\s<>"\']+',
        value
    )

    if url_match:
        return url_match.group(0)

    return value

# =====================================================
# LOAD
# =====================================================

with open(
    INPUT_FILE,
    "r",
    encoding="utf-8"
) as f:

    data = json.load(f)

TOTAL_PRODUCTS = len(data)

print(
    f"Loaded {TOTAL_PRODUCTS} products"
)

final_results = []

products_processed = 0

# =====================================================
# PROCESS
# =====================================================

for product_index, product in enumerate(
    data,
    start=1
):

    if product_index < START_FROM_PRODUCT:
        continue

    print(
        "\n"
        + "=" * 100
    )

    print(
        f"PRODUCT "
        f"{product_index}"
        f"/"
        f"{TOTAL_PRODUCTS}"
    )

    item_code = product.get(
        "ITEM_CODE"
    )

    print(
        f"ITEM_CODE: "
        f"{item_code}"
    )

    candidate_urls = product.get(
        "candidate_urls",
        []
    )

    candidate_summary = []

    for idx, candidate in enumerate(
        candidate_urls,
        start=1
    ):

        candidate_summary.append({

            "candidate_id":
                idx,

            "title":
                candidate.get(
                    "title",
                    ""
                ),

            "url":
                clean_url(
                    candidate.get(
                        "url",
                        ""
                    )
                ),

            "image_url":
                clean_url(
                    candidate.get(
                        "image_url",
                        ""
                    )
                ),

            "page_title":
                candidate.get(
                    "page_title",
                    ""
                ),

            "clean_text":
                candidate.get(
                    "clean_text",
                    ""
                ),

            "ocr":
                candidate.get(
                    "ocr",
                    {}
                ),

            "image_match_score":
                candidate.get(
                    "image_match_score",
                    ""
                )

        })

    prompt = f"""
You are a Product Truth Verification Agent.

Your task:

Select ONE best URL for the product.

Expected Product:

ITEM_CODE:
{product.get("ITEM_CODE","")}

BRAND:
{product.get("BRAND","")}

RETAILER_DESC:
{product.get("RETAILER_DESC","")}

SEARCH_QUERY:
{product.get("SEARCH_QUERY","")}

Candidate Evidence:

{json.dumps(candidate_summary, indent=2)}

Instructions:

1. Evaluate all candidate URLs.

2. Use:
   - page_title
   - clean_text
   - OCR
   - image_match_score

3. Prefer:

   exact brand

   exact variant

   exact product type

   exact pack size

   OCR relevance

4. Ignore noisy URLs.

5. Combine page evidence
   and image evidence.

6. Produce one final
   consolidated product
   description.

7. Explain WHY the URL
   was chosen.

Return JSON only.

{{
    "selected_candidate_id": 1,

    "best_url": "",

    "image_url": "",

    "final_match_score": 100,

    "final_product_description": "",

    "reasoning": ""
}}
"""

    try:

        response = client.complete(

            model=
            "hack-fest-gpt-5.6-luna",

            messages=[

                UserMessage(

                    content=[
                        {
                            "type":
                            "text",

                            "text":
                            prompt
                        }
                    ]

                )

            ],

            headers={
                "Authorization":
                api_key
            }

        )

        raw_response = (

            response
            .choices[0]
            .message
            .content

        )

        try:

            result = json.loads(
                raw_response
            )

        except Exception:

            result = {

                "parse_error":
                    True,

                "raw_response":
                    raw_response

            }

        final_row = {

            "ITEM_CODE":
                product.get(
                    "ITEM_CODE"
                ),

            "BRAND":
                product.get(
                    "BRAND"
                ),

            "RETAILER_DESC":
                product.get(
                    "RETAILER_DESC"
                ),

            "SEARCH_QUERY":
                product.get(
                    "SEARCH_QUERY"
                ),

            "BEST_URL":
                result.get(
                    "best_url",
                    ""
                ),

            "IMAGE_URL":
                result.get(
                    "image_url",
                    ""
                ),

            "FINAL_MATCH_SCORE":
                result.get(
                    "final_match_score",
                    ""
                ),

            "FINAL_PRODUCT_DESCRIPTION":
                result.get(
                    "final_product_description",
                    ""
                ),

            "REASONING":
                result.get(
                    "reasoning",
                    ""
                )

        }

        final_results.append(
            final_row
        )

        print(
            f"Selected URL:"
        )

        print(
            final_row[
                "BEST_URL"
            ]
        )

    except Exception as ex:

        print(ex)

        final_results.append({

            "ITEM_CODE":
                product.get(
                    "ITEM_CODE"
                ),

            "BRAND":
                product.get(
                    "BRAND"
                ),

            "RETAILER_DESC":
                product.get(
                    "RETAILER_DESC"
                ),

            "SEARCH_QUERY":
                product.get(
                    "SEARCH_QUERY"
                ),

            "BEST_URL":
                "",

            "IMAGE_URL":
                "",

            "FINAL_MATCH_SCORE":
                "",

            "FINAL_PRODUCT_DESCRIPTION":
                "",

            "REASONING":
                str(ex)

        })

    products_processed += 1

    # =================================================
    # CHECKPOINT
    # =================================================

    if (
        products_processed
        %
        CHECKPOINT_EVERY
        ==
        0
    ):

        with open(
            OUTPUT_FILE,
            "w",
            encoding="utf-8"
        ) as f:

            json.dump(
                final_results,
                f,
                indent=2,
                ensure_ascii=False
            )

        print(
            "\nCHECKPOINT SAVED"
        )

# =====================================================
# FINAL SAVE
# =====================================================

with open(
    OUTPUT_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        final_results,
        f,
        indent=2,
        ensure_ascii=False
    )

print(
    "\n"
    + "=" * 100
)

print(
    f"Products Processed: "
    f"{products_processed}"
)

print(
    f"Saved: "
    f"{OUTPUT_FILE}"
)

Loaded 412 products

PRODUCT 1/412
ITEM_CODE: 515000000
Selected URL:
https://www.superdrug.com/toiletries/dental/whitening-kits/brilliant-whitening-1-week-charcoal-kit/p/773567

PRODUCT 2/412
ITEM_CODE: 519000000
Selected URL:
https://www.youtube.com/shorts/5ot85VpzFNE&ntb=1?msockid=ed5f3a15b58c11f1a0c0f0b7d328162a

PRODUCT 3/412
ITEM_CODE: 55753033
Selected URL:


PRODUCT 4/412
ITEM_CODE: 2130865
Selected URL:
https://pricee.com/colgate-products-list

PRODUCT 5/412
ITEM_CODE: 9764265
Selected URL:
https://www.desertcart.com.au/products/905320-binaca-fastblast-breath-spray-peppermint-0-5-fl-oz-6

PRODUCT 6/412
ITEM_CODE: 4688813
Selected URL:
https://www.ebay.com/itm/168686583656?msockid=22d4743f8ae963311f4863e48b9862df

PRODUCT 7/412
ITEM_CODE: 509000000
Selected URL:
https://www.pointmeds.com/product/ultradex-one-go-mouthwash-on-the-go-sachets-pack-of-10/

PRODUCT 8/412
ITEM_CODE: 9315593
Selected URL:
https://www.cheapestinindia.com/price/spraymintt-fresh-breath

PRODUCT 9/412
ITEM

In [14]:
len(final_results)

412

In [15]:
final_results[:3]

[{'ITEM_CODE': 515000000,
  'BRAND': 'BRILLIANT (LI & FUNG)',
  'RETAILER_DESC': 'brilliant teeth whitening 1 week charcoal kit brandbank',
  'SEARCH_QUERY': 'Brilliant 1 week charcoal teeth whitening kit',
  'BEST_URL': 'https://www.superdrug.com/toiletries/dental/whitening-kits/brilliant-whitening-1-week-charcoal-kit/p/773567',
  'IMAGE_URL': '',
  'FINAL_MATCH_SCORE': 100,
  'FINAL_PRODUCT_DESCRIPTION': 'Brilliant 1 Week Charcoal Teeth Whitening Kit by Brilliant (Li & Fung), a one-week tooth whitening kit.',
  'REASONING': 'Candidate 1 is the strongest match because its title and clean text exactly identify the Brilliant Whitening 1 Week Charcoal Kit, matching the brand, charcoal variant, product type, and one-week pack size. Candidate 2 confirms the Brilliant whitening brand and one-week kit format through OCR, but it does not specify the charcoal variant and appears to represent a different generic tooth whitening kit. Therefore, candidate 1 is selected despite limited page access

In [16]:
count_product_url = 0
for i in final_results:
    if i['BEST_URL'] == '':
        count_product_url += 1
print(count_product_url)

53


In [17]:
count_image_url = 0
for i in final_results:
    if i['IMAGE_URL'] == '':
        count_image_url += 1
print(count_image_url)

144


In [19]:
import pandas as pd

In [27]:

qa = pd.read_excel(
    "search_queries_qa.xlsx"
)

print(qa.shape)
print(qa.columns.tolist())

(412, 24)
['ITEM_CODE', 'NAN_KEY', 'EXTERNAL_CODE', 'COUNTRY', 'RETAILER_DESC', 'RETAILER', 'BRAND', 'SEARCH_QUERY', 'PRODUCT_URL', 'REASONING', 'MODULE', 'GLOBAL_INTERSPACE_CLAIM', 'GLOBAL_CONSUMER_LIFESTAGE_CLAIM', 'GLOBAL_PACKAGING', 'GLOBAL_IF_MEDICATED', 'GLOBAL_PERCENTAGE_NATURAL_INGREDIENTS', 'GLOBAL_IF_WITH_SENSITIVE_CLAIM', 'GLOBAL_ORAL_CARE_FUNCTION', 'GLOBAL_IF_WITH_FLUORIDE', 'GLOBAL_FLAVOUR_FRAGRANCE_INGREDIENT_GROUP', 'GLOBAL_METHOD_OF_APPLICATION_DISPENSE', 'GLOBAL_PACKAGING_MATERIAL', 'GLOBAL_DESCRIPTIVE_SIZE_OF_TOOTHBRUSH_HEAD_CLAIM', 'GLOBAL_BRISTLE_STRENGTH_CLAIM']


In [25]:
qa.isnull().sum()

ITEM_CODE                                             0
NAN_KEY                                               0
EXTERNAL_CODE                                         0
COUNTRY                                               0
RETAILER_DESC                                         0
RETAILER                                              0
BRAND                                                 0
SEARCH_QUERY                                          0
PRODUCT_URL                                         412
REASONING                                           412
MODULE                                              412
GLOBAL_INTERSPACE_CLAIM                             412
GLOBAL_CONSUMER_LIFESTAGE_CLAIM                     412
GLOBAL_PACKAGING                                    412
GLOBAL_IF_MEDICATED                                 412
GLOBAL_PERCENTAGE_NATURAL_INGREDIENTS               412
GLOBAL_IF_WITH_SENSITIVE_CLAIM                      412
GLOBAL_ORAL_CARE_FUNCTION                       

In [26]:
qa["MODULE"].value_counts()

Series([], Name: count, dtype: int64)

In [28]:
dev = pd.read_excel(
    "dev_with_search_queries.xlsx"
)
dev["MODULE"].value_counts()

MODULE
TOOTH CLEANING - FOAM/GEL/LIQUID/PASTE (NATURAL TEETH)                          133
MOUTHWASH/ORAL RINSES/ORAL RINSE ANTISEPTICS - FOAM/GEL/LIQUID - MULTI DOSE      94
TOOTHBRUSHES - MANUAL - REGULAR                                                  52
TOOTHBRUSHES - ELECTRIC - COMPLETE PACK                                          38
TOOTHBRUSHES - ELECTRIC - REFILL HEADS                                           13
DENTAL FLOSS/TAPE - PRE CUT PIECES/SINGLES                                        9
BREATH FRESHENERS - GEL/LIQUID - MULTI DOSE                                       8
TOOTH STAIN REMOVERS - FOAM/GEL/LIQUID/PASTE - MULTI DOSE                         8
BREATH FRESHENERS - CAPSULES/GUM/LOZENGES/TABLETS                                 8
TOOTH CLEANING - GUM/TABLETS (NATURAL TEETH)                                      7
DENTURE CLEANSERS - TABLETS                                                       6
TOOTH CLEANING - FINGER GLOVES/WIPES - DISPOSABLE                    

In [36]:
import json
import pandas as pd

# =====================================================
# FILES
# =====================================================

DEV_FILE = "search_queries_qa.xlsx"

PRODUCT_TRUTH_FILE = "final_product_truth_qa.json"

OUTPUT_FILE = "dev_training_dataset_qa.xlsx"

# =====================================================
# LOAD DEV
# =====================================================

dev_df = pd.read_excel(
    DEV_FILE
)

print(
    f"DEV rows: {len(dev_df)}"
)

# =====================================================
# LOAD PRODUCT TRUTH
# =====================================================

with open(
    PRODUCT_TRUTH_FILE,
    "r",
    encoding="utf-8"
) as f:

    truth_data = json.load(f)

truth_df = pd.DataFrame(
    truth_data
)

print(
    f"Product Truth rows: "
    f"{len(truth_df)}"
)

# =====================================================
# VALIDATE
# =====================================================

if len(dev_df) != len(truth_df):

    raise ValueError(
        f"Row count mismatch. "
        f"DEV={len(dev_df)} "
        f"TRUTH={len(truth_df)}"
    )

# =====================================================
# ADD COLUMNS
# =====================================================

dev_df[
    "FINAL_PRODUCT_DESCRIPTION"
] = truth_df[
    "FINAL_PRODUCT_DESCRIPTION"
]

dev_df[
    "Reasoning"
] = truth_df[
    "REASONING"
]
 


# =====================================================
# SAVE
# =====================================================

dev_df.to_excel(
    OUTPUT_FILE,
    index=False
)

print(
    "\n=================================="
)

print(
    f"Rows Written: "
    f"{len(dev_df)}"
)

print(
    f"Saved: "
    f"{OUTPUT_FILE}"
)

DEV rows: 412
Product Truth rows: 412

Rows Written: 412
Saved: dev_training_dataset_qa.xlsx


In [37]:
import pandas as pd

from azure.ai.inference.models import (
    UserMessage
)

# =====================================================
# LOAD
# =====================================================

FILE = "dev_training_dataset_qa.xlsx"

OUTPUT_FILE = (
    "dev_module_validation_qa.xlsx"
)


# =====================================================
# LOAD MODULE LIST FROM DEV
# =====================================================

dev_df = pd.read_excel(
    "dev_with_search_queries.xlsx"
)

valid_modules = sorted(

    dev_df["MODULE"]
    .dropna()
    .astype(str)
    .str.strip()
    .unique()
    .tolist()

)

module_text = "\n".join(
    valid_modules
)






df = pd.read_excel(FILE)

# =====================================================
# PREDICT
# =====================================================

predicted_modules = []

correct = 0

total = 0

for idx, row in df.iterrows():

    description = str(
        row[
            "FINAL_PRODUCT_DESCRIPTION"
        ]
    )

    # actual_module = str(
    #     row[
    #         "MODULE"
    #     ]
    # )

    prompt = f"""
You are an NIQ Oral Care Module Classifier.

Choose exactly ONE module
from the following list.

Available Modules:

{module_text}

Product Description:

{description}

Return only the module name.
"""

    try:

        response = client.complete(

            model=
            "hack-fest-gpt-5.6-luna",

            messages=[

                UserMessage(

                    content=[

                        {
                            "type":
                            "text",

                            "text":
                            prompt
                        }

                    ]

                )

            ],

            headers={

                "Authorization":
                api_key

            }

        )

        predicted = (

            response
            .choices[0]
            .message
            .content

        ).strip()

    except Exception:

        predicted = ""

    predicted_modules.append(
        predicted
    )

    if (
        predicted
        ==
        actual_module
    ):
        correct += 1

    total += 1

    print(
        f"{idx+1}/{len(df)} | "
        f"{predicted}"
    )

# =====================================================
# SAVE
# =====================================================

df["MODULE"] = predicted_modules

df.to_excel(
    OUTPUT_FILE,
    index=False
)

print(
    "\n===================================="
)

print(
    f"Total Rows : {len(df)}"
)

print(
    f"Saved : "
    f"{OUTPUT_FILE}"
)

1/412 | TOOTH STAIN REMOVERS - STRIPS/TRAYS/WIPES
2/412 | TOOTH STAIN REMOVERS - STRIPS/TRAYS/WIPES
3/412 | DENTURE CLEANSERS - FOAM/GEL/LIQUID/PASTE
4/412 | TOOTH CLEANING - FOAM/GEL/LIQUID/PASTE (NATURAL TEETH)
5/412 | BREATH FRESHENERS - GEL/LIQUID - MULTI DOSE
6/412 | BREATH FRESHENERS - HERBAL COMPOUNDS
7/412 | MOUTHWASH/ORAL RINSES/ORAL RINSE ANTISEPTICS - FOAM/GEL/LIQUID - SINGLE DOSE
8/412 | BREATH FRESHENERS - GEL/LIQUID - MULTI DOSE
9/412 | DENTAL ACCESSORIES - TOOTHPICKS - MANUAL - DISPOSABLE
10/412 | TOOTH CLEANING - GUM/TABLETS (NATURAL TEETH)
11/412 | BREATH FRESHENERS - GEL/LIQUID - MULTI DOSE
12/412 | TOOTH CLEANING - FOAM/GEL/LIQUID/PASTE (NATURAL TEETH)
13/412 | DENTAL FLOSS/TAPE - PRE CUT PIECES/SINGLES
14/412 | DENTURE CLEANSERS - TABLETS
15/412 | DENTAL FLOSS/TAPE - PRE CUT PIECES/SINGLES
16/412 | TOOTHBRUSHES - MANUAL - REGULAR
17/412 | TOOTH STAIN REMOVERS - FOAM/GEL/LIQUID/PASTE - MULTI DOSE
18/412 | BREATH FRESHENERS - CAPSULES/GUM/LOZENGES/TABLETS
19/412 | TOO

In [38]:
import pandas as pd

# =====================================================
# FILES
# =====================================================

RULES_FILE = "product_truth.xlsx"

# =====================================================
# LOAD SHEETS
# =====================================================

char_value_list = pd.read_excel(
    RULES_FILE,
    sheet_name="char_value_list"
)

char_guidelines = pd.read_excel(
    RULES_FILE,
    sheet_name="char_guidelines"
)

print(
    f"char_value_list rows: "
    f"{len(char_value_list)}"
)

print(
    f"char_guidelines rows: "
    f"{len(char_guidelines)}"
)

# =====================================================
# STANDARDIZE COLUMN NAMES
# =====================================================

char_value_list.columns = [

    c.strip()
    .lower()
    .replace(" ", "_")

    for c in char_value_list.columns

]

char_guidelines.columns = [

    c.strip()
    .lower()
    .replace(" ", "_")

    for c in char_guidelines.columns

]

# =====================================================
# VIEW STRUCTURE
# =====================================================

print("\nchar_value_list columns:")

print(
    char_value_list.columns.tolist()
)

print("\nchar_guidelines columns:")

print(
    char_guidelines.columns.tolist()
)

# =====================================================
# EXAMPLE
# =====================================================

print(
    "\nSample value list rows:"
)

print(
    char_value_list.head()
)

print(
    "\nSample guideline rows:"
)

print(
    char_guidelines.head()
)

char_value_list rows: 195
char_guidelines rows: 196

char_value_list columns:
['category', 'module', 'characteristic', 'open_close', 'binary', 'possible_values', 'notes']

char_guidelines columns:
['haleon_category', 'module_name', 'characteristics_name', 'guidelines']

Sample value list rows:
      category                            module  \
0  ORAL HEALTH  ORAL HYGIENE - COMBINATION PACKS   
1  ORAL HEALTH  ORAL HYGIENE - COMBINATION PACKS   
2  ORAL HEALTH  ORAL HYGIENE - COMBINATION PACKS   
3  ORAL HEALTH   TONGUE CLEANING - BRUSH/SCRAPER   
4  ORAL HEALTH   TONGUE CLEANING - BRUSH/SCRAPER   

                                     characteristic  open_close binary  \
0                     GLOBAL BRISTLE STRENGTH CLAIM       Close      N   
1  GLOBAL DESCRIPTIVE SIZE OF TOOTHBRUSH HEAD CLAIM       Close      N   
2             GLOBAL PERCENTAGE NATURAL INGREDIENTS  Open-ended      N   
3                     GLOBAL BRISTLE STRENGTH CLAIM       Close      N   
4  GLOBAL DESCRIPTIVE 

In [41]:
import json
import re

# =====================================================
# FILES
# =====================================================

PRODUCT_TRUTH_FILE = "final_product_truth_qa.json"

OCR_FILE = "ocr_verified_products_filtered_qa.json"

OUTPUT_FILE = (
    "final_product_truth_with_clean_text_qa.json"
)

# =====================================================
# URL CLEANER
# =====================================================

def clean_url(value):

    value = str(value)

    href_match = re.search(
        r'href="([^"]+)"',
        value
    )

    if href_match:
        return href_match.group(1)

    url_match = re.search(
        r'https?://[^\s<>"\']+',
        value
    )

    if url_match:
        return url_match.group(0)

    return value

# =====================================================
# LOAD FILES
# =====================================================

with open(
    PRODUCT_TRUTH_FILE,
    "r",
    encoding="utf-8"
) as f:

    product_truth = json.load(f)

with open(
    OCR_FILE,
    "r",
    encoding="utf-8"
) as f:

    ocr_data = json.load(f)

# =====================================================
# BUILD ITEM_CODE LOOKUP
# =====================================================

ocr_lookup = {}

for product in ocr_data:

    item_code = str(
        product.get(
            "ITEM_CODE",
            ""
        )
    )

    ocr_lookup[item_code] = product

# =====================================================
# COPY CLEAN_TEXT
# =====================================================

matched = 0

for product in product_truth:

    item_code = str(
        product.get(
            "ITEM_CODE",
            ""
        )
    )

    best_url = clean_url(
        product.get(
            "BEST_URL",
            ""
        )
    )

    product["CLEAN_TEXT"] = ""

    if item_code not in ocr_lookup:
        continue

    candidate_urls = ocr_lookup[
        item_code
    ].get(
        "candidate_urls",
        []
    )

    for candidate in candidate_urls:

        candidate_url = clean_url(
            candidate.get(
                "url",
                ""
            )
        )

        if candidate_url == best_url:

            product[
                "CLEAN_TEXT"
            ] = candidate.get(
                "clean_text",
                ""
            )

            matched += 1

            break

# =====================================================
# SAVE
# =====================================================

with open(
    OUTPUT_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        product_truth,
        f,
        indent=2,
        ensure_ascii=False
    )

# =====================================================
# SUMMARY
# =====================================================

print(
    "\n======================================"
)

print(
    f"Products: "
    f"{len(product_truth)}"
)

print(
    f"Matched Clean Text: "
    f"{matched}"
)

print(
    f"Saved: "
    f"{OUTPUT_FILE}"
)


Products: 412
Matched Clean Text: 284
Saved: final_product_truth_with_clean_text_qa.json


In [44]:
import pandas as pd
import json

# =====================================================
# FILES
# =====================================================

QA_FILE = "dev_module_validation_qa.xlsx"

OCR_FILE = "ocr_verified_products_filtered_qa.json"

OUTPUT_FILE = (
    "dev_module_validation_qa_with_clean_text.xlsx"
)

# =====================================================
# LOAD QA
# =====================================================

qa_df = pd.read_excel(
    QA_FILE
)

# =====================================================
# LOAD OCR DATA
# =====================================================

with open(
    OCR_FILE,
    "r",
    encoding="utf-8"
) as f:

    ocr_data = json.load(f)

# =====================================================
# BUILD ITEM_CODE -> CLEAN_TEXT LOOKUP
# =====================================================

clean_text_lookup = {}

for product in ocr_data:

    item_code = str(
        product.get(
            "ITEM_CODE",
            ""
        )
    )

    # best_evidence = product.get(
    #     "best_image_evidence",
    #     {}
    # )

    # clean_text = best_evidence.get(
    #     "clean_text",
    #     ""
    # )

    best_evidence = product.get(
    "best_image_evidence"
    )

    if best_evidence is None:

        clean_text = ""

    else:

        clean_text = best_evidence.get(
            "clean_text",
            ""
        )






    clean_text_lookup[
        item_code
    ] = clean_text

# =====================================================
# ADD CLEAN_TEXT COLUMN
# =====================================================

qa_df["ITEM_CODE"] = (
    qa_df["ITEM_CODE"]
    .astype(str)
)

qa_df["CLEAN_TEXT"] = qa_df[
    "ITEM_CODE"
].map(
    clean_text_lookup
)

qa_df["CLEAN_TEXT"] = qa_df[
    "CLEAN_TEXT"
].fillna(
    ""
)

# =====================================================
# SAVE
# =====================================================

qa_df.to_excel(
    OUTPUT_FILE,
    index=False
)

# =====================================================
# SUMMARY
# =====================================================

matched = (
    qa_df["CLEAN_TEXT"]
    .ne("")
    .sum()
)

print(
    "\n===================================="
)

print(
    f"Rows : {len(qa_df)}"
)

print(
    f"Clean Text Matched : {matched}"
)

print(
    f"Saved : {OUTPUT_FILE}"
)


Rows : 412
Clean Text Matched : 381
Saved : dev_module_validation_qa_with_clean_text.xlsx


In [59]:
import pandas as pd

from azure.ai.inference.models import (
    UserMessage
)

# =====================================================
# FILES
# =====================================================

QA_FILE = "dev_module_validation_qa_with_clean_text.xlsx"

RULES_FILE = "product_truth.xlsx"

OUTPUT_FILE = "global_predictions_qa.xlsx"

# =====================================================
# LOAD DATA
# =====================================================

qa_df = pd.read_excel(
    QA_FILE
)




# =====================================================
# FORCE GLOBAL_* AS TEXT
# =====================================================

global_columns = [

    c

    for c in qa_df.columns

    if c.startswith(
        "GLOBAL_"
    )

]

for col in global_columns:

    qa_df[col] = (

        qa_df[col]

        .astype(str)

        .replace(
            "nan",
            ""
        )

    )







char_value_list = pd.read_excel(
    RULES_FILE,
    sheet_name="char_value_list"
)

char_guidelines = pd.read_excel(
    RULES_FILE,
    sheet_name="char_guidelines"
)

# =====================================================
# STANDARDIZE RULES
# =====================================================

char_value_list.columns = [

    c.strip()
    .lower()
    .replace(" ", "_")

    for c in char_value_list.columns

]

char_guidelines.columns = [

    c.strip()
    .lower()
    .replace(" ", "_")

    for c in char_guidelines.columns

]

# =====================================================
# MODULE -> CHARACTERISTICS MAP
# =====================================================

module_char_map = {}

for _, row in char_value_list.iterrows():

    module = str(
        row["module"]
    ).strip()

    if module not in module_char_map:

        module_char_map[module] = []

    module_char_map[module].append({

        "characteristic":
            str(
                row["characteristic"]
            ).strip(),

        "open_close":
            str(
                row["open_close"]
            ).strip(),

        "binary":
            str(
                row["binary"]
            ).strip(),

        "possible_values":
            str(
                row["possible_values"]
            ).strip()

    })

# =====================================================
# GLOBAL COLUMNS
# =====================================================

global_columns = [

    c

    for c in qa_df.columns

    if c.startswith(
        "GLOBAL_"
    )

]

# =====================================================
# PROCESS PRODUCTS
# =====================================================

for idx, row in qa_df.iterrows():

    module = str(
        row.get(
            "MODULE",
            ""
        )
    ).strip()

    if not module:

        print(
            f"\n[{idx+1}] No Module"
        )

        continue

    if module not in module_char_map:

        print(
            f"\n[{idx+1}] Unknown Module"
        )

        print(module)

        continue

    item_code = str(
        row.get(
            "ITEM_CODE",
            ""
        )
    )

    description = str(
        row.get(
            "FINAL_PRODUCT_DESCRIPTION",
            ""
        )
    )

    clean_text = str(
        row.get(
            "CLEAN_TEXT",
            ""
        )
    )

    reasoning = str(
        row.get(
            "Reasoning",
            ""
        )
    )

    print(
        "\n"
        + "=" * 80
    )

    print(
        f"PRODUCT "
        f"{idx+1}/{len(qa_df)}"
    )

    print(
        f"ITEM_CODE: "
        f"{item_code}"
    )

    print(
        f"MODULE: "
        f"{module}"
    )

    populated_values = {}

    # -------------------------------------------------
    # CHARACTERISTICS FOR MODULE
    # -------------------------------------------------

    for char_info in module_char_map[module]:
        characteristic = (
            char_info[
                "characteristic"
            ]
        )

        value_rows = char_value_list[

            (
                char_value_list[
                    "module"
                ]
                ==
                module
            )

            &

            (
                char_value_list[
                    "characteristic"
                ]
                ==
                characteristic
            )

        ]

        guideline_rows = char_guidelines[

            (
                char_guidelines[
                    "module_name"
                ]
                ==
                module
            )

            &

            (
                char_guidelines[
                    "characteristics_name"
                ]
                ==
                characteristic
            )

        ]

        possible_values = ""

        if len(value_rows):

            possible_values = (
                value_rows
                .iloc[0]
                .get(
                    "possible_values",
                    ""
                )
            )

        guideline = ""

        if len(guideline_rows):

            guideline = (
                guideline_rows
                .iloc[0]
                .get(
                    "guidelines",
                    ""
                )
            )

        prompt = f"""
You are an NIQ Oral Care Characteristic Classification Expert.

Predicted Module:

{module}

Characteristic:

{characteristic}

Restriction Type:

{char_info['open_close']}

Binary:

{char_info['binary']}

Possible Values:

{possible_values}

Guideline:

{guideline}

Final Product Description:

{description}

Page Evidence:

{clean_text}

Supporting Reasoning:

{reasoning}

Instructions:

1. Use only:
   - Final Product Description
   - Page Evidence
   - Supporting Reasoning

2. Follow guideline exactly.

3. If characteristic is Close:
   return exactly one value
   from possible values.

4. Apply defaults.

5. Return ONLY the final value.
"""

        try:

            response = client.complete(

                model=
                "hack-fest-gpt-5.6-luna",

                messages=[

                    UserMessage(

                        content=[

                            {
                                "type":
                                "text",

                                "text":
                                prompt
                            }

                        ]

                    )

                ],

                headers={
                    "Authorization":
                    api_key
                }

            )

            predicted_value = (

                response
                .choices[0]
                .message
                .content

            ).strip()

        except Exception as ex:

            predicted_value = ""

            print(
                f"ERROR: "
                f"{characteristic}"
            )

            print(ex)

        # -----------------------------------------
        # WRITE INTO EXISTING GLOBAL_* COLUMN
        # -----------------------------------------

        target_col = (characteristic.strip()
                      .replace(
                            " ",
                            "_"
                        )
                      )



        if target_col in qa_df.columns:

            qa_df.at[
                idx,
                target_col
            ] = predicted_value

            if predicted_value:

                populated_values[
                    target_col
                ] = predicted_value

    # =================================================
    # SHOW ONLY POPULATED GLOBALS
    # =================================================

    print(
        "\nPredicted GLOBAL_*"
    )

    if populated_values:

        for key, value in populated_values.items():

            print(
                f"{key}"
                f" = "
                f"{value}"
            )

    else:

        print(
            "No values populated"
        )

    # =================================================
    # CHECKPOINT AFTER EVERY PRODUCT
    # =================================================

    qa_df.to_excel(
        OUTPUT_FILE,
        index=False
    )

    print(
        "\nCHECKPOINT SAVED"
    )

# =====================================================
# FINAL SAVE
# =====================================================

qa_df.to_excel(
    OUTPUT_FILE,
    index=False
)

print(
    "\n"
    + "=" * 80
)

print(
    "COMPLETED"
)

print(
    f"Rows Processed: "
    f"{len(qa_df)}"
)

print(
    f"Saved: "
    f"{OUTPUT_FILE}"
)


PRODUCT 1/412
ITEM_CODE: 515000000
MODULE: TOOTH STAIN REMOVERS - STRIPS/TRAYS/WIPES

Predicted GLOBAL_*
GLOBAL_PERCENTAGE_NATURAL_INGREDIENTS = NOT STATED

CHECKPOINT SAVED

PRODUCT 2/412
ITEM_CODE: 519000000
MODULE: TOOTH STAIN REMOVERS - STRIPS/TRAYS/WIPES

Predicted GLOBAL_*
GLOBAL_PERCENTAGE_NATURAL_INGREDIENTS = NOT STATED

CHECKPOINT SAVED

PRODUCT 3/412
ITEM_CODE: 55753033
MODULE: DENTURE CLEANSERS - FOAM/GEL/LIQUID/PASTE

Predicted GLOBAL_*
GLOBAL_PERCENTAGE_NATURAL_INGREDIENTS = NOT STATED

CHECKPOINT SAVED

PRODUCT 4/412
ITEM_CODE: 2130865
MODULE: TOOTH CLEANING - FOAM/GEL/LIQUID/PASTE (NATURAL TEETH)

Predicted GLOBAL_*
GLOBAL_CONSUMER_LIFESTAGE_CLAIM = NO CLAIM
GLOBAL_IF_WITH_FLUORIDE = WITHOUT FLUORIDE
GLOBAL_IF_WITH_SENSITIVE_CLAIM = WITHOUT SENSITIVE CLAIM
GLOBAL_ORAL_CARE_FUNCTION = FRESHENING
GLOBAL_PERCENTAGE_NATURAL_INGREDIENTS = NOT STATED
GLOBAL_PACKAGING = PACKET
GLOBAL_PACKAGING_MATERIAL = PLASTIC

CHECKPOINT SAVED

PRODUCT 5/412
ITEM_CODE: 9764265
MODULE: BREA

In [63]:
import pandas as pd

# =====================================================
# FILES
# =====================================================

INPUT_FILE = "global_predictions_qa.xlsx"

OUTPUT_FILE = "global_predictions_qa_final.xlsx"

# =====================================================
# LOAD
# =====================================================

df = pd.read_excel(INPUT_FILE)

# =====================================================
# COPY Reasoning -> REASONING
# =====================================================

if "Reasoning" in df.columns:

    df["REASONING"] = df["Reasoning"]

# =====================================================
# DROP COLUMNS
# =====================================================

cols_to_drop = [

    "Reasoning",

    "FINAL_PRODUCT_DESCRIPTION",

    "CLEAN_TEXT",

    "SEARCH_QUERY"

]

existing_cols = [

    c

    for c in cols_to_drop

    if c in df.columns

]

df = df.drop(
    columns=existing_cols
)

# =====================================================
# SAVE
# =====================================================

df.to_excel(
    OUTPUT_FILE,
    index=False
)

print(
    "\n===================================="
)

print(
    f"Rows: {len(df)}"
)

print(
    f"Saved: {OUTPUT_FILE}"
)


Rows: 412
Saved: global_predictions_qa_final.xlsx


In [65]:
import pandas as pd
import json
import re

# =====================================================
# FILES
# =====================================================

INPUT_FILE = "global_predictions_qa_final.xlsx"

PRODUCT_TRUTH_FILE = "final_product_truth_qa.json"

OUTPUT_FILE = "global_predictions_qa_final.xlsx"

# =====================================================
# URL CLEANER
# =====================================================

def extract_url(value):

    if pd.isna(value):
        return ""

    value = str(value)

    href_match = re.search(
        r'href="([^"]+)"',
        value
    )

    if href_match:
        return href_match.group(1)

    url_match = re.search(
        r'https?://[^\s<>"\']+',
        value
    )

    if url_match:
        return url_match.group(0)

    return value

# =====================================================
# LOAD EXCEL
# =====================================================

df = pd.read_excel(
    INPUT_FILE
)

# =====================================================
# LOAD JSON
# =====================================================

with open(
    PRODUCT_TRUTH_FILE,
    "r",
    encoding="utf-8"
) as f:

    final_results = json.load(f)

# =====================================================
# ITEM_CODE -> URL LOOKUP
# =====================================================

url_lookup = {}

for row in final_results:

    item_code = str(
        row.get(
            "ITEM_CODE",
            ""
        )
    ).strip()

    best_url = extract_url(
        row.get(
            "BEST_URL",
            ""
        )
    )

    url_lookup[item_code] = best_url

# =====================================================
# FILL PRODUCT_URL
# =====================================================

df["ITEM_CODE"] = (
    df["ITEM_CODE"]
    .astype(str)
    .str.strip()
)

matched = 0

product_urls = []

for item_code in df["ITEM_CODE"]:

    url = url_lookup.get(
        item_code,
        ""
    )

    if url:
        matched += 1

    product_urls.append(url)

df["PRODUCT_URL"] = product_urls

# =====================================================
# SAVE
# =====================================================

df.to_excel(
    OUTPUT_FILE,
    index=False
)

# =====================================================
# SUMMARY
# =====================================================

print("\n====================================")

print(
    f"Rows: {len(df)}"
)

print(
    f"URLs Filled: {matched}"
)

print(
    f"Saved: {OUTPUT_FILE}"
)


Rows: 412
URLs Filled: 367
Saved: global_predictions_qa_final.xlsx
